In [1]:
!pip install git+https://github.com/huggingface/transformers torch accelerate bitsandbytes langchain
!pip install evaluate
!pip install -U sentence-transformers
!pip install langchain_community
!pip install peft
!pip install evaluate
!pip install rouge_score
!pip install bert_score
!pip install faiss-gpu

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-ganhotti
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-ganhotti
  Resolved https://github.com/huggingface/transformers to commit 7f79a97399bb52aad8460e1da2f36577d5dccfed
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 919.0 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 12.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.6/983.6 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141

In [2]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from langchain.llms import HuggingFacePipeline # This should work after the upgrade
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
device = 'cuda' if torch.cuda.is_available() else 'cpu'

2024-07-13 07:04:24.846222: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-13 07:04:24.846335: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-13 07:04:24.952519: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from peft import PeftModel
from langchain.retrievers import EnsembleRetriever

In [4]:
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
)

In [5]:
model_name = "filipealmeida/Mistral-7B-Instruct-v0.1-sharded"
peft_model_id = "ProElectro07/Projectx3x3"
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             torch_dtype=torch.bfloat16,
                                             quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


pytorch_model.bin.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

pytorch_model-00001-of-00008.bin:   0%|          | 0.00/1.89G [00:00<?, ?B/s]

pytorch_model-00002-of-00008.bin:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

In [6]:
model = PeftModel.from_pretrained(model, peft_model_id)
model = model.merge_and_unload()

adapter_config.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/peft/tuners/lora/bnb.py:325: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [7]:
from bert_score import score

In [8]:
!pip install rouge
from evaluate import load
rouge = load('rouge')
bertscore = load('bertscore')
bleu = load('bleu')

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [9]:
import pandas as pd
df = pd.read_csv("/kaggle/input/covidd/covid_abstracts.csv")
df = df["abstract"]
train = pd.read_csv("/kaggle/input/covidd/train.csv")

In [45]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
    separators=['\n\n']
)

In [46]:
docs = [Document(page_content=entry) for entry in df]

docs = text_splitter.split_documents(docs)

In [12]:
embedding_model = SentenceTransformerEmbeddings(model_name='BAAI/bge-large-zh-v1.5')

/opt/conda/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/439k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [47]:
faiss_db = FAISS.from_documents(docs, embedding_model)

In [48]:
faiss_retriever = faiss_db.as_retriever(
    search_kwargs={'k': 2}
)

In [15]:
!pip install rank_bm25

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [49]:
from langchain.schema import Document
from langchain_community.retrievers import BM25Retriever
from typing import List

def custom_preprocessing_func(text: str) -> List[str]: # Change the input type to str
    return text.split()

bm25_retriever = BM25Retriever.from_texts(
    [doc.page_content for doc in docs],
    preprocess_func=custom_preprocessing_func
)
bm25_retriever.k = 2

In [50]:
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

In [51]:
query = "mujhe pet mein bahut dard hain, iska kya ilaaj hain?"


text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1.,
    top_k=50,
    top_p=.9,
    repetition_penalty=.75,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=False
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

In [52]:
response = mistral_llm.generate([query], max_new_tokens=200)

/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [53]:
response

LLMResult(generations=[[Generation(text='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n')]], llm_output=None, run=[RunInfo(run_id=UUID('56550bd8-3266-47d7-a7ef-be09cec9af54'))])

In [54]:

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

In [55]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.0, 1.0]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=0.65,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=True,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)


In [56]:
response = RAG_chain({"query": train["Queries "][0]})

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [57]:
response

{'query': '1. "Corona ke symptoms kya hote hain?"',
 'result': '\n<s>[INST]\nYou are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient\'s specific question.\nDo not ask the patient how you can assist them or request more information unless absolutely necessary.\nUnderstand each word very carefully to give an answer, do not assume anything about the query of the patient.\n\n\nKey points to remember:\n1. Always respond in English, even if the patient uses Hindi or mixed language.\n2. Provide a direct answer to the patient\'s question immediately.\n3. Use simple, non-technical language that patients can easily understand.\n4. Give practical, actionable advice when appropriate.\n5. Be reassuring and empathetic, but honest about health concerns.\n6. If you\'re unsure about a specific treatment or diagnosis, recommend consulting a doctor

In [58]:
r = '\n<s>[INST]\nYou are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient\'s specific question.\nDo not ask the patient how you can assist them or request more information unless absolutely necessary.\nUnderstand each word very carefully to give an answer, do not assume anything about the query of the patient.\n\n\nKey points to remember:\n1. Always respond in English, even if the patient uses Hindi or mixed language.\n2. Provide a direct answer to the patient\'s question immediately.\n3. Use simple, non-technical language that patients can easily understand.\n4. Give practical, actionable advice when appropriate.\n5. Be reassuring and empathetic, but honest about health concerns.\n6. If you\'re unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.\n\nHere are two examples of good responses:\n\nPatient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?\nDoctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.\n\nPatient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?\nDoctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it\'s still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.\n\nNow, please respond to the patient\'s query below in a similar manner, in English:\n\nPatient\'s question: 1. "Corona ke symptoms kya hote hain?"\n\nAdditional context (if available):\nCOVID-19 has been a considerable major stress factor and has led to a great increase in psychological illnesses  Literature related to spirituality and philosophy can provide solace in times like these  We have taken excerpts from Bhagwad Gita which is a spiritual and philosophical scripture  existential philosopher Albert Camus s La Peste and the myth of Sisyphus and writings of the philosopher Immanule Kant  These literary masterpieces can provide a wealth of guidance and solace to both the medical fraternity and affected people and their caregivers   We tell ourselves that pestilence is a mere bogy of the mind  a bad dream that will pass away  But it doesn t always pass away and from one bad dream to another  it is men who pass away   -Albert Camus\n\nCoronavirus disease 2019  COVID-19  is an infectious diseases caused by severe acute respiratory syndrome coronavirus 2  SARS-CoV-2   Now  it is pandemic over the world  SARS-CoV-2 often causes a  cytokine storm  in people with COVID-19  causing inflammatory lung damage and pneumonia  which eventually leads to death  Glucagon like peptide-1  GLP-1  is well known as an incretin hormone responsible for regulation of blood glucose through its receptor  Beyond glycemic control  GLP-1 receptor agonists  GLP-1RAs  have promising anti-inflammatory actions in human and rodent pathological models  Recent studies proved that GLP-1RAs attenuate pulmonary inflammation  reduce cytokine production  and preserve lung function in mice and rats with experimental lung injury  Moreover  a thickened pulmonary vascular wall  an important characteristic of pulmonary arterial hypertension  PAH  was observed in the autopsy lung tissue of a COVID-19 patient  Thus GLP-1RAs may be a novel therapeutic strategy for combating this pandemic specifically for patient characteristics of PHA after COVID-19 infection\n\nBACKGROUND  Coronavirus disease 2019  COVID-19  is known to cause not only respiratory but also neuropsychiatric symptoms  which are assumed to be derived from a cytokine storm and its effects on the central nervous systems  Patients with COVID-19 who develop severe respiratory symptoms are known to show severe neuropsychiatric symptoms such as cerebrovascular disease and encephalopathy  However  the detailed clinical courses of patients with neuropsychiatric symptoms caused by mild or asymptomatic COVID-19 remain poorly understood  Here  we present a case of COVID-19 who presented with severe and prolonged neuropsychiatric symptoms subsequent to mild respiratory symptoms  CASE PRESENTATION  A 55-year-old female with COVID-19 accompanied by mild respiratory symptoms showed delusion  psychomotor excitement  and poor communication ability during quarantine outside the hospital  Considering her diminished respiratory symptoms  her neuropsychiatric symptoms were initially regarded as psychogenic reactions  However  as she showed progressive disturbance of consciousness accompanied by an abnormal electroencephalogram  she was diagnosed with post-COVID-19 encephalopathy  Although her impaired consciousness and elevated cytokine level improved after steroid pulse therapy  several neuropsychiatric symptoms  including a loss of concentration  unsteadiness while walking  and fatigue  remained  CONCLUSIONS  This case suggests the importance of both recognizing that even apparently mild COVID-19-related respiratory symptoms can lead to severe and persistent neuropsychiatric symptoms  and elucidating the mechanisms  treatment  and long-term course of COVID-19-related neuropsychiatric symptoms in the future\n\nBACKGROUND  Depressive and anxiety symptoms are widespread among adolescents today  creating a large social problem  However  few previous studies have addressed depression and anxiety among adolescents in Chinese cohorts  The aim of this study was to evaluate the prevalence of and risk factors for depressive and anxiety symptoms among Chinese middle school adolescent students in the post-COVID-19 era  METHODS  A total of 22 380 middle school students from Jiangsu Province were surveyed online  and their general demographic data were collected  The Patient Health Questionnaire-9  PHQ-9  was used to assess depressive symptoms  and the seven-item Generalized Anxiety Disorder  GAD-7  scale was used to measure anxiety symptoms  RESULTS  Of these participants  aged 12-17 years   25 6  had depressive symptoms  26 9  had anxiety symptoms  and 20 6  had a combination of depression and anxiety symptoms  The prevalence of depressive symptoms was higher in female adolescents  27 6   than in male adolescents  23 7   chi 2    45 479  P   0 000   and the proportion with anxiety symptoms was higher among female adolescents  28 6   than among male adolescents  25 4   chi 2    29 390  P   0 000   Furthermore  binary logistic regression analysis showed that gender  region  and parental relationship were significantly associated with depressive symptoms among adolescents  while age  gender  region  and parental relationship were significantly associated with anxiety symptoms  CONCLUSIONS  Our findings demonstrated that the prevalence of reported depressive and anxiety symptoms in Chinese adolescents are high  Female gender  urban region  and poor parental relationship may be risk factors for depressive and anxiety symptoms  Furthermore  policy makers  schools  and families need to pay more attention to the psychological health of adolescents  develop response plans and take early intervention measures to reduce the prevalence of adolescent depression and anxiety\n\nProvide a relevant, caring, and informative response:\n[/INST] </s>\n\nHello! How can I help you today? Is there something specific you would like me to address regarding COVID-19 symptoms? Remember that COVID-19 can cause various symptoms, including fever, dry cough, shortness of breath, body aches, headache, new loss of taste or smell, sore throat, congestion or runny nose, nausea or vomiting, diarrhea, fatigue, or muscle pain. It\'s essential to monitor your symptoms closely and seek medical attention if any of these symptoms worsen, especially if you experience difficulty breathing. Additionally, practice good hygiene habits, wear a mask in public spaces, maintain physical distancing, and avoid close contact with sick individuals. Stay informed about local guidelines and restrictions to prevent spreading the virus further. Please let me know if you have any other questions or concerns.'

In [59]:
prompt_tokens = tokenizer(r, return_tensors="pt", truncation=True)
num_prompt_tokens = len(r)
num_prompt_tokens

8527

In [60]:
# ensemble_retriever = EnsembleRetriever(
#     retrievers=[bm25_retriever, faiss_retriever],
#     weights=[.90, .10]
# )

# text_generation_pipeline = pipeline(
#     task="text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     temperature=0.65,
#     top_k=50,
#     top_p=.9,
#     repetition_penalty=1.20,
#     return_full_text=False,
#     max_new_tokens=200,
#     truncation=True,
#     do_sample=True
# )

# mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

# prompt_template = """
# <s>[INST]
# You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
# Do not ask the patient how you can assist them or request more information unless absolutely necessary.
# Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


# Key points to remember:
# 1. Always respond in English, even if the patient uses Hindi or mixed language.
# 2. Provide a direct answer to the patient's question immediately.
# 3. Use simple, non-technical language that patients can easily understand.
# 4. Give practical, actionable advice when appropriate.
# 5. Be reassuring and empathetic, but honest about health concerns.
# 6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

# Here are two examples of good responses:

# Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
# Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

# Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
# Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

# Now, please respond to the patient's query below in a similar manner, in English:

# Patient's question: {question}

# Additional context (if available):
# {context}

# Provide a relevant, caring, and informative response:
# [/INST] </s>
# """

# Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

# RAG_chain = RetrievalQA.from_chain_type(
#     llm=mistral_llm,
#     retriever=ensemble_retriever,
#     chain_type_kwargs={"prompt": Prompt_Template}
# )

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1.70 seconds, 5.89 sentences/sec
ROUGE scores: {'rouge1': 0.03229510833779567, 'rouge2': 0.008924173937513127, 'rougeL': 0.021733410524791316, 'rougeLsum': 0.028784247772198443}
BERT scores: {'precision': 0.7686226963996887, 'recall': 0.8487666249275208, 'f1': 0.8066974878311157}
BLEU score: {'bleu': 0.0038207327765048316, 'precisions': [0.017297064523074716, 0.005674064497593909, 0.0020125062890821535, 0.0010789038337049557], 'brevity_penalty': 1.0, 'length_ratio': 44.94516129032258, 'translation_length': 13933, 'reference_length': 310}


In [63]:
predictions

[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': '\n<s>[INST]\nYou are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient\'s specific question.\nDo not ask the patient how you can assist them or request more information unless absolutely necessary.\nUnderstand each word very carefully to give an answer, do not assume anything about the query of the patient.\n\n\nKey points to remember:\n1. Always respond in English, even if the patient uses Hindi or mixed language.\n2. Provide a direct answer to the patient\'s question immediately.\n3. Use simple, non-technical language that patients can easily understand.\n4. Give practical, actionable advice when appropriate.\n5. Be reassuring and empathetic, but honest about health concerns.\n6. If you\'re unsure about a specific treatment or diagnosis, recommend consulting a do

In [65]:
prompt_tokens = tokenizer(r, return_tensors="pt", truncation=True)
num_prompt_tokens = len(r)
print(num_prompt_tokens)

ValueError: text input must be of type `str` (single example), `List[str]` (batch or single pretokenized example) or `List[List[str]]` (batch of pretokenized examples).

In [66]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.90, .10]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=0.65,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.60 seconds, 16.75 sentences/sec
ROUGE scores: {'rouge1': 0.14764011675197253, 'rouge2': 0.03927246577481469, 'rougeL': 0.10429089102175965, 'rougeLsum': 0.11134177419152605}
BERT scores: {'precision': 0.833823561668396, 'recall': 0.881606936454773, 'f1': 0.8566789627075195}
BLEU score: {'bleu': 0.017823476843431912, 'precisions': [0.10401188707280833, 0.021706586826347306, 0.00980392156862745, 0.004559270516717325], 'brevity_penalty': 1.0, 'length_ratio': 4.341935483870968, 'translation_length': 1346, 'reference_length': 310}


In [67]:
predictions

[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': '\nHello! How can I help you today?'}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': '\n"Yes, wearing a mask is always recommended during physical activity, regardless of the type of mask or level of intensity. Wearing a mask may help filter out respiratory droplets that can carry viruses, including those responsible for COVID-19. However, it\'s essential to choose a mask that fits properly and allows for adequate ventilation to avoid discomfort and potential negative impacts on physical performance. Additionally, it\'s crucial to maintain proper hygiene practices before, during, and after exercise to further minimize the risk of transmission."'}],
 [{'query': '3. "Vaccine ke side effects ke baare mein batao."',
   'result': "\nThank you for reaching out to me. As per your question, here are some potential side effects of COVID-19 vaccination that may occur:\n\n* Fever\n* Fatigue\n* Headache\n

In [92]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.0, 1.0]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

In [93]:
response = RAG_chain({"query": "Corona ke symptoms kya hote hain?"})

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [94]:
response

{'query': 'Corona ke symptoms kya hote hain?',
 'result': "Coronavirus, commonly known as COVID-19, causes a range of symptoms including fever, dry cough, fatigue, shortness of breath, muscle pain, sore throat, headache, congestion or runny nose, vomiting or diarrhea, abdominal pain, and loss of sense of taste or smell. It typically spreads through respiratory droplets when an infected person talks, sneezes, or coughs. In some people, the virus may cause mild or moderate symptoms, while others may develop severe illness with pneumonia and multi-organ failure, potentially leading to death. Vaccines such as Pfizer-BioNTech, Moderna, and Johnson & Johnson have shown high efficacy in preventing symptomatic COVID-19 infections and reducing mortality rates. It's essential to maintain personal protective practices such as wearing masks, practicing social distancing, washing hands frequently, avoiding large gatherings,"}

In [95]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.0, 1.0]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.53 seconds, 19.04 sentences/sec
ROUGE scores: {'rouge1': 0.17929747190904916, 'rouge2': 0.050393673112053596, 'rougeL': 0.12802941548620383, 'rougeLsum': 0.13091651558602047}
BERT scores: {'precision': 0.8416226506233215, 'recall': 0.8898512125015259, 'f1': 0.8649503588676453}
BLEU score: {'bleu': 0.03164241468921528, 'precisions': [0.12263436790310371, 0.03813882532418002, 0.018447348193697154, 0.011618900077459334], 'brevity_penalty': 1.0, 'length_ratio': 4.2612903225806456, 'translation_length': 1321, 'reference_length': 310}


In [96]:
predictions

[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': "\nSymptoms of COVID-19:\nThe main symptoms of COVID-19 include fever, dry cough, shortness of breath, headache, muscle pain, body aches, sore throat, new loss of taste or smell, congestion or runny nose, nausea or vomiting, diarrhea, fatigue or weakness, dizziness, rapid breathing or wheezing, difficulty sleeping, joint or muscle aches, chest pain, confusion or disorientation, seizures, mood changes, hallucinations, depression, anxiety, mood swings, or change in mental state. However, some people may not experience any noticeable symptoms at all, and others might have very mild ones. It is important to monitor any symptoms and contact your healthcare provider if they worsen or become more frequent. Remember, you don't have to wait until you feel ill to stay safe! Wear a mask in public places, practice social dist"}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': 'Hello! Thank you for reaching 

In [100]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.0, 1.0]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.57 seconds, 17.56 sentences/sec
ROUGE scores: {'rouge1': 0.136217421193687, 'rouge2': 0.02762090780963189, 'rougeL': 0.09836493090380971, 'rougeLsum': 0.10219480391492235}
BERT scores: {'precision': 0.8324161767959595, 'recall': 0.8739219903945923, 'f1': 0.8525810241699219}
BLEU score: {'bleu': 0.009970325496556583, 'precisions': [0.1015748031496063, 0.01507936507936508, 0.004, 0.0016129032258064516], 'brevity_penalty': 1.0, 'length_ratio': 4.096774193548387, 'translation_length': 1270, 'reference_length': 310}


In [101]:
predictions

[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': '\nHello, my name is Dr. XYZ and I am here to help you. Is there something particular you would like me to address regarding COVID-19? Please feel free to ask any questions you might have.'}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': '\n"Yes, wearing a mask is an essential precautionary measure against COVID-19."\n\nIt is recommended that everyone wear a mask in public spaces to protect themselves and others from getting infected with COVID-19. Masks should cover the nose and mouth and be worn correctly to ensure maximum protection. Wearing a mask helps to block droplets containing the virus from spreading into the air and onto surfaces around us. Additionally, wearing a mask can help to reduce the spread of other respiratory infections. It is important to remember to wash your hands regularly and practice safe physical distancing practices as well. If you have any further questions or con

In [102]:
import time

In [104]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.0, 1.0]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.

Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []
timings = []


for p, r in zip(train["Queries "], train["Response"]):
    start_time = time.time()
    response = RAG_chain({"query": p})
    end_time = time.time()
    
    response_time = end_time - start_time
    timings.append(response_time)
    
    references.append([r])
    predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)
bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

# Print metrics and timings
print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)
print("Response times:", timings)
print("Average response time:", sum(timings) / len(timings))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.54 seconds, 18.59 sentences/sec
ROUGE scores: {'rouge1': 0.17477127538508613, 'rouge2': 0.050696909227429116, 'rougeL': 0.1182251941247984, 'rougeLsum': 0.12124668285642236}
BERT scores: {'precision': 0.8336722254753113, 'recall': 0.8866362571716309, 'f1': 0.8592937588691711}
BLEU score: {'bleu': 0.02364355416500127, 'precisions': [0.11432604093154553, 0.032693674484719264, 0.012884753042233358, 0.006488824801730353], 'brevity_penalty': 1.0, 'length_ratio': 4.570967741935484, 'translation_length': 1417, 'reference_length': 310}
Response times: [23.205947160720825, 27.5274338722229, 29.594778537750244, 23.875938177108765, 24.012385606765747, 20.72567844390869, 36.2977819442749, 23.624138832092285, 25.969911336898804, 27.093540906906128]
Average response time: 26.19275348186493


In [105]:
predictions

[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': '"Koi bhi bhalo ki jata hai nao chikna ho sakti hai jab bhi chikna ho sakti hai." ("Anything happens, no matter what, either good or bad.") It\'s important to maintain hope and positivity during difficult times. Remember that seeking help is a sign of strength and taking care of yourself is essential. Keep practicing self-care by getting enough sleep, eating nutritious meals, staying active, and managing any underlying conditions. Also, stay connected with loved ones virtually or through phone calls. Lastly, don\'t hesitate to reach out to healthcare professionals if you feel overwhelmed or need support.'}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': 'Yes, it is important to wear a mask. Masks help to prevent the spread of airborne particles, including viruses, by trapping them before they reach others. Wearing a mask can also protect the wearer from inhaling harmful particles that may be pr

In [106]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.0, 1.0]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.

Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []
timings = []


for p, r in zip(train["Queries "], train["Response"]):
    start_time = time.time()
    response = RAG_chain({"query": p})
    end_time = time.time()
    
    response_time = end_time - start_time
    timings.append(response_time)
    
    references.append([r])
    predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)
bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

# Print metrics and timings
print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)
print("Response times:", timings)
print("Average response time:", sum(timings) / len(timings))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.51 seconds, 19.70 sentences/sec
ROUGE scores: {'rouge1': 0.13717528194286174, 'rouge2': 0.035676842320862504, 'rougeL': 0.09585964664180836, 'rougeLsum': 0.09688332730943724}
BERT scores: {'precision': 0.8346943855285645, 'recall': 0.8745092153549194, 'f1': 0.8540046811103821}
BLEU score: {'bleu': 0.018096287013591866, 'precisions': [0.10434056761268781, 0.02356902356902357, 0.008488964346349746, 0.005136986301369863], 'brevity_penalty': 1.0, 'length_ratio': 3.8645161290322583, 'translation_length': 1198, 'reference_length': 310}
Response times: [24.36531090736389, 24.662771701812744, 21.81059741973877, 25.78510594367981, 16.41364073753357, 28.941489458084106, 34.73080492019653, 10.715008974075317, 26.353055000305176, 27.232762098312378]
Average response time: 24.10105471611023


In [107]:
predictions

[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': "Hello there! Thank you for reaching out. Let me explain what COVID-19 symptoms look like. Symptoms usually start with feeling feverish, having chills and coughing with sputum. Sometimes patients experience other common symptoms such as sore throat, headache, body aches, and new loss of taste or smell. Some people might also feel nausea, vomiting, or diarrhea. However, some individuals may not show any symptoms at all. It's crucial to keep monitoring your symptoms closely and seeking medical attention promptly if your condition deteriorates, particularly if you have difficulty breathing or chest tightness. Stay healthy and safe, and don't hesitate to reach out if you have further questions or concerns."}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': '"Yes, wearing a mask is definitely recommended in all public settings where physical distancing cannot be maintained, including at work, in groc

In [108]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.0, 1.0]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.49 seconds, 20.44 sentences/sec
ROUGE scores: {'rouge1': 0.1600210868022967, 'rouge2': 0.03640264163420885, 'rougeL': 0.11131502006544214, 'rougeLsum': 0.11084168514619486}
BERT scores: {'precision': 0.8369234800338745, 'recall': 0.8758220672607422, 'f1': 0.8557637929916382}
BLEU score: {'bleu': 0.022432153556307736, 'precisions': [0.1231203007518797, 0.028462998102466792, 0.012452107279693486, 0.005802707930367505], 'brevity_penalty': 1.0, 'length_ratio': 3.432258064516129, 'translation_length': 1064, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': '\nDear patient,\nThank you for coming in to see me. As you know, COVID-19 can cause various symptoms, some of which include fever and cough. It is important to note that these symptoms can also be indicative of other conditions, so if you suspect you have COVID-19, it is best to get tested to confirm your diagnosis. In addition to seeking medical attention if needed, there are things you can do to help manage any potential symptoms you might be experiencing. First and foremost, make sure you are getting plenty of sleep, eating nutritious food, and staying physically active. Additionally, practicing stress reduction techniques such as meditation or yoga can help keep your immune system strong. Finally, it is essential to maintain personal protective practices such as wearing a mask and washing your hands regularly to prevent further spread of the virus.'}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   '

In [109]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.1, .9]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.53 seconds, 18.92 sentences/sec
ROUGE scores: {'rouge1': 0.17229876518117382, 'rouge2': 0.04674110704350917, 'rougeL': 0.11016022432179524, 'rougeLsum': 0.11786144728302675}
BERT scores: {'precision': 0.8389102220535278, 'recall': 0.8833528757095337, 'f1': 0.8603607416152954}
BLEU score: {'bleu': 0.026460528715415488, 'precisions': [0.12141652613827993, 0.034013605442176874, 0.0137221269296741, 0.00865051903114187], 'brevity_penalty': 1.0, 'length_ratio': 3.825806451612903, 'translation_length': 1186, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': 'Symptoms of corona?\n\nThe common symptoms of COVID-19 include fever, dry cough, difficulty breathing, muscle pain, headache, sore throat, and loss of taste or smell. Additionally, some people with COVID-19 experience nausea, vomiting, diarrhea, fatigue, and body aches. It is important to note that some individuals with COVID-19 may not exhibit any symptoms at all, yet still be contagious. If you suspect that you or someone close to you may have COVID-19, I encourage you to speak with your healthcare provider and self-isolate until receiving proper testing and guidance. Remember to stay informed about best practices for preventing the spread of the virus, such as washing hands regularly, wearing face masks, practicing physical distancing, and avoiding contact with sick individuals.'}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': "\nYes, wearing a mask is indeed essential for preventing the s

In [110]:
predictionsensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.2, .8]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.53 seconds, 18.97 sentences/sec
ROUGE scores: {'rouge1': 0.1904278323025731, 'rouge2': 0.06514610215349856, 'rougeL': 0.13676374124554336, 'rougeLsum': 0.1364703797372778}
BERT scores: {'precision': 0.8455418348312378, 'recall': 0.8983732461929321, 'f1': 0.87102210521698}
BLEU score: {'bleu': 0.0405014240115978, 'precisions': [0.12582781456953643, 0.04595997034840623, 0.0268857356235997, 0.01730624529721595], 'brevity_penalty': 1.0, 'length_ratio': 4.383870967741935, 'translation_length': 1359, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': '"Corona ke symptoms kya hote hain?" (What are the symptoms of corona?)\nThe common symptoms of COVID-19 include fever, dry cough, fatigue, body aches, headache, new loss of taste or smell, sore throat, congestion or runny nose, nausea or vomiting, diarrhea, abdominal pain, or difficulty swallowing. Some people experience mild symptoms, while others may have no noticeable symptoms. However, anyone experiencing symptoms should contact their healthcare provider for evaluation and testing. It\'s essential to monitor these symptoms daily, practice self-care measures, such as resting and drinking plenty of fluids, and maintain social distancing measures to prevent further transmission of the virus.'}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': "\nYes, wearing a mask is definitely recommended in many situations, including public spaces and workplaces. This can help prevent the spread of viruses a

In [111]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.3, .7]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.62 seconds, 16.10 sentences/sec
ROUGE scores: {'rouge1': 0.1704837250466977, 'rouge2': 0.050186248559206184, 'rougeL': 0.11913471419299465, 'rougeLsum': 0.11769615469009455}
BERT scores: {'precision': 0.8244602084159851, 'recall': 0.8877707719802856, 'f1': 0.8548341989517212}
BLEU score: {'bleu': 0.02603379142231367, 'precisions': [0.10818505338078292, 0.03369175627240143, 0.01444043321299639, 0.008727272727272728], 'brevity_penalty': 1.0, 'length_ratio': 4.532258064516129, 'translation_length': 1405, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': "The most common symptoms of COVID-19 include fever, cough, shortness of breath, fatigue, sore throat, body aches, headache, new loss of taste or smell, congestion or runny nose, nausea or vomiting, diarrhea, muscle pain or weakness, chills, and confusion. It's essential to monitor any change in your symptoms and consult a healthcare provider if any symptom worsens, especially difficulty breathing, chest pain, loss of appetite, dehydration, or confusion."}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': '\nPatient\'s question: 2. "Kya mask pehenna abhi bhi zaroori hai?"\n\nResponse: Yes, wearing a mask is still important to protect yourself and others against COVID-19, even if you have already received a vaccine. Masks help prevent the spread of the virus through respiratory droplets, especially when combined with other preventive measures like physical distancing, hand hygiene, and avoiding la

In [112]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.4, .6]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.61 seconds, 16.50 sentences/sec
ROUGE scores: {'rouge1': 0.17591982406673645, 'rouge2': 0.04926119582723357, 'rougeL': 0.12146143327882682, 'rougeLsum': 0.12676332035952323}
BERT scores: {'precision': 0.8396846055984497, 'recall': 0.8944774866104126, 'f1': 0.8661583065986633}
BLEU score: {'bleu': 0.02017862732136933, 'precisions': [0.1121311475409836, 0.0297029702970297, 0.0106312292358804, 0.0046822742474916385], 'brevity_penalty': 1.0, 'length_ratio': 4.919354838709677, 'translation_length': 1525, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': '\n"Corona ke symptoms kya hote hain?"\n\nThe most common symptoms of COVID-19 include a fever above 100.4°F (38°C), cough, shortness of breath, headache, sore throat, muscle pain, fatigue, body or muscle aches, new loss of taste or smell, congestion, runny nose, nausea or vomiting, diarrhea, loss of appetite, upset stomach, abdominal pain, feeling dizzy, lightheaded, confused, or unable to focus, chest tightness, shortness of breath or rapid heartbeat, weakness or muscle pain, chills, sweating, or clotting problems. It\'s essential to keep in mind that some people with COVID-19 don\'t experience any symptoms. However, if you have symptoms of COVID-19, it\'s crucial to contact your healthcare provider immediately to determine whether you need'}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': "It depends on where you live and what kind of masks are readily available. Generally, it's recommended 

In [113]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.5, .5]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.59 seconds, 16.99 sentences/sec
ROUGE scores: {'rouge1': 0.1781328985472077, 'rouge2': 0.04441823881924632, 'rougeL': 0.1277703191684541, 'rougeLsum': 0.13202997104267528}
BERT scores: {'precision': 0.8386870622634888, 'recall': 0.8949463963508606, 'f1': 0.8658736944198608}
BLEU score: {'bleu': 0.016952285640015455, 'precisions': [0.11619462599854757, 0.027066569129480616, 0.008843036109064112, 0.002969561989606533], 'brevity_penalty': 1.0, 'length_ratio': 4.441935483870968, 'translation_length': 1377, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': 'Hello, thank you for reaching out. Doctors are here to help. It sounds like you want to know what COVID-19 means and whether it has any symptoms. To start with, let me explain that COVID-19 stands for "coronavirus disease 2019." It is an illness caused by the coronavirus called SARS-CoV-2. Some people with COVID-19 experience mild or no symptoms, while others might have severe ones. Common symptoms include fever, cough, difficulty breathing, fatigue, headache, muscle pain or soreness, and chills. These can last anywhere from a couple of days to weeks. Keep in mind, however, that having some or all of these symptoms does not necessarily mean you have COVID-19. It could be something else entirely, so it\'s essential to consult a doctor or test for COVID-19.\nIn terms of prevention, there are many steps you can take to'}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': "\nYes, wearing a mask is de

In [114]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.6, .4]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.54 seconds, 18.59 sentences/sec
ROUGE scores: {'rouge1': 0.16401599459540678, 'rouge2': 0.04122048197159732, 'rougeL': 0.1197338174806793, 'rougeLsum': 0.12293802718163757}
BERT scores: {'precision': 0.8417550921440125, 'recall': 0.8918789029121399, 'f1': 0.8660192489624023}
BLEU score: {'bleu': 0.01164469474860791, 'precisions': [0.1075725406935598, 0.023521026372059873, 0.005025125628140704, 0.0014461315979754157], 'brevity_penalty': 1.0, 'length_ratio': 4.558064516129032, 'translation_length': 1413, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': "```markdown\nI apologize for any confusion, but let me clarify a couple of things. Firstly, COVID-19 is a viral infection that primarily affects the respiratory system, but it can also affect other parts of the body, including the brain. Symptoms such as delusions, psychomotor excitability, and cognitive difficulties are typically referred to as neurological symptoms and can occur alongside or without respiratory symptoms. While COVID-19 itself does not directly cause brain damage, it is thought that some of the symptoms may be due to inflammation within the brain caused by the body's immune response to the virus. It's important to note that everyone's experience with COVID-19 is different, and some people may not develop any significant brain or cognitive issues at all. Therefore, it's essential to monitor any changes in behavior or cognition closely and consult with a healthcare professional if you notice any concern

In [115]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.7, .3]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.53 seconds, 19.01 sentences/sec
ROUGE scores: {'rouge1': 0.1899236519243881, 'rouge2': 0.0642842614647168, 'rougeL': 0.14168789987678626, 'rougeLsum': 0.1430657546678218}
BERT scores: {'precision': 0.8496672511100769, 'recall': 0.8966153860092163, 'f1': 0.8724220395088196}
BLEU score: {'bleu': 0.036652189956056846, 'precisions': [0.13065755764304013, 0.046511627906976744, 0.022589052997393572, 0.013146362839614373], 'brevity_penalty': 1.0, 'length_ratio': 3.7774193548387096, 'translation_length': 1171, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': "Hello, thank you for reaching out to me. I am happy to help clarify any questions you may have regarding COVID-19. Let me first say that the main symptoms of COVID-19 include fever, cough, shortness of breath, body aches, sore throat, headache, new loss of taste or smell, and congestion or runny nose. However, some individuals may experience additional symptoms such as neurological issues. It's essential to note that COVID-19 is an ongoing pandemic, and research into its impacts continues. If you experience any symptoms, please consult with a healthcare professional immediately. Remember to wear a mask, practice social distancing, and wash your hands frequently to stay safe. Stay healthy!"}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': '\nYes, wearing a mask is recommended during physical activity, including exercise. Wearing a mask can help prevent the spread of respiratory droplets, especi

In [116]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.8, .2]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.52 seconds, 19.22 sentences/sec
ROUGE scores: {'rouge1': 0.1515263664438723, 'rouge2': 0.03322858607737808, 'rougeL': 0.10235795984677298, 'rougeLsum': 0.10301874779333327}
BERT scores: {'precision': 0.8350154757499695, 'recall': 0.8835388422012329, 'f1': 0.858405590057373}
BLEU score: {'bleu': 0.019083232120107804, 'precisions': [0.11164274322169059, 0.02491961414790997, 0.009724473257698542, 0.004901960784313725], 'brevity_penalty': 1.0, 'length_ratio': 4.04516129032258, 'translation_length': 1254, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': '\nHello! Thank you for reaching out. What would you like me to help you with today? Remember, I am here to listen and support you. Feel free to share any questions or concerns you might have.'}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': "\nYes, wearing a mask while exercising is indeed essential to protect oneself and others from spreading the virus. Wearing a mask may affect one's physical ability to breathe properly, leading to higher perceived exertion levels, and reduced performance during intense exercises. Therefore, it is crucial to use a properly fitting mask and take breaks whenever needed to avoid discomfort and fatigue. Additionally, exercising with a partner or joining virtual classes can provide motivation and accountability to maintain consistent workouts. Remember, staying active and healthy is crucial for fighting off illness and managing potential complications related to

In [117]:
######

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.9, .1]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.58 seconds, 17.29 sentences/sec
ROUGE scores: {'rouge1': 0.19406839453605934, 'rouge2': 0.04912513826572141, 'rougeL': 0.13348598643321302, 'rougeLsum': 0.1331552574989598}
BERT scores: {'precision': 0.8434749841690063, 'recall': 0.8961032629013062, 'f1': 0.8689075708389282}
BLEU score: {'bleu': 0.02206226151721017, 'precisions': [0.11979166666666667, 0.03373313343328336, 0.01283987915407855, 0.0045662100456621], 'brevity_penalty': 1.0, 'length_ratio': 4.335483870967742, 'translation_length': 1344, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': "\nCovid Ke Symptoms Kya Hota Hai?\n\nHello! It's important to know the signs and symptoms of Covid-19 so that you can stay safe and healthy. Some common symptoms include fever, cough, shortness of breath, body aches, headache, sore throat, nausea, vomiting, diarrhea, fatigue, and difficulty sleeping. Additionally, some people with Covid-19 experience neurological symptoms, such as confusion, hallucinations, seizures, and movement disorders. It's essential to monitor your symptoms closely and consult with a healthcare professional if any symptoms become severe or if there are any changes in your condition. Remember, taking preventative measures, such as washing hands frequently and wearing masks, can help prevent the spread of Covid-19. Stay informed and stay safe!"}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': "Yes, it is generally recommended to wear a mask while engaging in physical activ

In [118]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[1, .0]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.60 seconds, 16.57 sentences/sec
ROUGE scores: {'rouge1': 0.14505978340147752, 'rouge2': 0.03191501629281661, 'rougeL': 0.10608934054192169, 'rougeLsum': 0.10826106786821671}
BERT scores: {'precision': 0.8286277055740356, 'recall': 0.882207989692688, 'f1': 0.8544970750808716}
BLEU score: {'bleu': 0.013362307872322534, 'precisions': [0.10119465917076599, 0.020523708421797595, 0.007127583749109052, 0.0021536252692031586], 'brevity_penalty': 1.0, 'length_ratio': 4.590322580645161, 'translation_length': 1423, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': "\nHello there, I'm glad you're reaching out. Let me clarify some things. First of all, COVID-19 symptoms are varied and can include physical symptoms like fever and dry cough, as well as neurological symptoms like confusion, hallucinations, seizures, and coma. It's essential to consult with a healthcare professional and have a thorough evaluation before determining what is going on with those symptoms. If you feel worried or stressed, remember that it's okay to reach out to friends, family, or support groups, who can help provide emotional support during challenging situations. And don't forget to take precautions against COVID-19, like washing hands regularly, wearing a mask indoors or in public spaces, and following social distancing guidelines. Keep up to date with accurate information regarding COVID-19 prevention, transmission, and treatment to ensure your health and well-being. Thank you for reaching out and taki

In [119]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.0, 1.]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.59 seconds, 16.87 sentences/sec
ROUGE scores: {'rouge1': 0.17834236318918364, 'rouge2': 0.05406701990345639, 'rougeL': 0.13309118439121825, 'rougeLsum': 0.13592221978237923}
BERT scores: {'precision': 0.8406184315681458, 'recall': 0.8967393636703491, 'f1': 0.8676789999008179}
BLEU score: {'bleu': 0.03063746089136221, 'precisions': [0.11902956785443518, 0.03972498090145149, 0.018475750577367205, 0.01008533747090768], 'brevity_penalty': 1.0, 'length_ratio': 4.254838709677419, 'translation_length': 1319, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': '\n"Corona Ke Symptoms Ka Hote Hain?"\n\nYes, there are many different symptoms that people can experience when they have contracted COVID-19. Some common symptoms include fever, dry cough, shortness of breath, sore throat, fatigue, muscle or body pain, headache, new loss of taste or smell, congestion or runny nose, nausea or vomiting, and diarrhea. Other less common symptoms can include skin rash, hair loss, and changes in mood. It\'s important to note that some people with COVID-19 may not exhibit any symptoms at all, and symptoms can vary widely between individuals. If you have any concerns about symptoms you are experiencing, or if you think you may have been exposed to someone with COVID-19, I would recommend contacting your healthcare provider for guidance and evaluation. In the meantime, it\'s best to self-isolate and monitor your symptoms closely'}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   

In [120]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.1, .9]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.57 seconds, 17.60 sentences/sec
ROUGE scores: {'rouge1': 0.19223976334678622, 'rouge2': 0.059583845552273404, 'rougeL': 0.1422219950292111, 'rougeLsum': 0.1448519512703607}
BERT scores: {'precision': 0.8423566818237305, 'recall': 0.8877439498901367, 'f1': 0.8643331527709961}
BLEU score: {'bleu': 0.02103657606010276, 'precisions': [0.11085450346420324, 0.02560124127230411, 0.0109460516028147, 0.006304176516942475], 'brevity_penalty': 1.0, 'length_ratio': 4.190322580645161, 'translation_length': 1299, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': '"Symptoms of COVID-19 include fever, cough, shortness of breath, body pain, headache, sore throat, congestion or runny nose, vomiting, diarrhea, new loss of taste or smell."'}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': "\nYes, wearing a mask is indeed recommended in most situations to help prevent the spread of viruses, including COVID-19. Wearing a mask protects others by preventing respiratory droplets from traveling through the air and onto surfaces, reducing the likelihood of transmission. Additionally, masks can offer some protection to the wearer by filtering out certain particles, including larger droplets. Therefore, it is always important to wear a mask in public settings where physical distancing may not be feasible, particularly indoors and in areas with poor ventilation. Remember to practice other preventive measures, such as washing hands frequently and avoiding close contact

In [121]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.2, .8]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.52 seconds, 19.08 sentences/sec
ROUGE scores: {'rouge1': 0.1569217611099668, 'rouge2': 0.04536499939280473, 'rougeL': 0.11525681671053295, 'rougeLsum': 0.11682970266369648}
BERT scores: {'precision': 0.8360971212387085, 'recall': 0.888563334941864, 'f1': 0.861430287361145}
BLEU score: {'bleu': 0.028002395747089803, 'precisions': [0.11303692539562923, 0.033409263477600606, 0.017597551644988524, 0.009252120277563608], 'brevity_penalty': 1.0, 'length_ratio': 4.280645161290322, 'translation_length': 1327, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': "The coronavirus disease 2019, commonly referred to as COVID-19, is caused by severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2). It affects the lungs and can sometimes cause a cytokine storm, leading to inflammatory lung damage, pneumonia, and potentially fatal complications. Some common symptoms of COVID-19 include fever, cough, shortness of breath, fatigue, body aches, headache, new loss of taste or smell, sore throat, congestion or runny nose, nausea or vomiting, diarrhea. However, it's important to note that some people with COVID-19 might not exhibit any noticeable signs or symptoms, making it crucial to monitor and report any changes in our health status promptly. To protect yourself from getting COVID-19, practice personal protective measures like washing hands frequently, wearing a mask ind"}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': "No matter what the reason may be, i

In [122]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.3, .7]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.58 seconds, 17.23 sentences/sec
ROUGE scores: {'rouge1': 0.15890996045975225, 'rouge2': 0.04827465415764657, 'rougeL': 0.11704516391115041, 'rougeLsum': 0.11891395676564828}
BERT scores: {'precision': 0.8329631686210632, 'recall': 0.8894730806350708, 'f1': 0.8602016568183899}
BLEU score: {'bleu': 0.026264778137417587, 'precisions': [0.1070110701107011, 0.03420074349442379, 0.017228464419475654, 0.007547169811320755], 'brevity_penalty': 1.0, 'length_ratio': 4.370967741935484, 'translation_length': 1355, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': 'I understand you would like to know what the symptoms of COVID-19 are. According to the Centers for Disease Control and Prevention (CDC), common symptoms of COVID-19 include fever, cough, shortness of breath, fatigue, body aches, headache, new loss of taste or smell, sore throat, congestion or runny nose, nausea or vomiting, diarrhea, difficulty sleeping, and difficulty concentrating. It is essential to note that some individuals with COVID-19 may not experience any symptoms at all, making it challenging to diagnose the disease. If you suspect you or someone else has COVID-19, it is crucial to contact a healthcare professional right away and discuss testing options. In the meantime, please practice preventative measures to protect yourself and others, such as wearing a mask, practicing physical distancing, washing hands frequently, avoiding crowds, and covering your mouth and nose when coughing or sne'}],
 [{'query': '

In [123]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.4, .6]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.51 seconds, 19.78 sentences/sec
ROUGE scores: {'rouge1': 0.151390379915108, 'rouge2': 0.0319322742503505, 'rougeL': 0.10068405363291003, 'rougeLsum': 0.10054306189260409}
BERT scores: {'precision': 0.8384807705879211, 'recall': 0.8847052454948425, 'f1': 0.8608558773994446}
BLEU score: {'bleu': 0.013006025996786836, 'precisions': [0.11423550087873462, 0.022163120567375887, 0.006261180679785331, 0.0018050541516245488], 'brevity_penalty': 1.0, 'length_ratio': 3.670967741935484, 'translation_length': 1138, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': '\nThe symptoms of coronavirus (COVID-19) include a range of conditions like fever, dry cough, headache, shortness of breath, body pain, nausea, vomiting, diarrhea, conjunctival congestion, dizziness, confusion, seizures, and neurological symptoms like hallucinations and coma. While some individuals may experience mild symptoms or no symptoms at all, others might suffer serious complications leading to death. Therefore, it is essential to monitor any new symptoms or changes in existing ones and seek prompt medical attention if needed, especially if there are signs of trouble breathing. Regular exercise, healthy eating, adequate sleep, avoiding close contact with sick people, wearing a mask, and following guidelines set forth by local authorities can help prevent the spread of COVID-19. In addition, getting vaccinated against the virus can help protect against severe infection and its potentially deadly consequences. Rem

In [124]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.5, .5]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.52 seconds, 19.24 sentences/sec
ROUGE scores: {'rouge1': 0.15095061135041504, 'rouge2': 0.033194310595820065, 'rougeL': 0.10496640879132688, 'rougeLsum': 0.11179484425908834}
BERT scores: {'precision': 0.8419631719589233, 'recall': 0.8868464231491089, 'f1': 0.8637107610702515}
BLEU score: {'bleu': 0.010306717443003897, 'precisions': [0.10804416403785488, 0.023052464228934817, 0.005608974358974359, 0.0008077544426494346], 'brevity_penalty': 1.0, 'length_ratio': 4.090322580645161, 'translation_length': 1268, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': "Dear patient,\nIt's normal to experience physical symptoms like fever and cough along with neurological symptoms such as delusions, motor problems, and cognitive difficulties following COVID-19. While many cases resolve without any complications, it is crucial to monitor any signs of progression, particularly changes in consciousness, such as confusion or hallucinations.\nIt is essential to note that the clinical course and severity of COVID-19 vary greatly across individuals, even those with seemingly mild symptoms. Therefore, understanding your condition is critical. Regular medical checkups and consultations can help identify any underlying issues early, allowing us to manage symptoms better and reduce potential risks."}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': 'Yes, wearing a face mask is generally recommended in most situations where physical distancing may not be possible, includi

In [125]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.6, .4]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.59 seconds, 17.05 sentences/sec
ROUGE scores: {'rouge1': 0.14631370273535993, 'rouge2': 0.03596899133489086, 'rougeL': 0.10325095100705076, 'rougeLsum': 0.11107655413871757}
BERT scores: {'precision': 0.830518364906311, 'recall': 0.8789944648742676, 'f1': 0.8540006875991821}
BLEU score: {'bleu': 0.016987217945809388, 'precisions': [0.10653409090909091, 0.027181688125894134, 0.00792507204610951, 0.00362844702467344], 'brevity_penalty': 1.0, 'length_ratio': 4.541935483870968, 'translation_length': 1408, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': "The patient's question could potentially refer to various aspects of Coronavirus disease 2019, also commonly referred to as COVID-19. As someone who specializes in diagnosing and treating COVID-19, here's my best attempt at providing a comprehensive, yet concise, response:\n\nFirstly, Coronavirus disease 2019, also called COVID-19, is an infectious disease caused by severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2). COVID-19 can manifest itself in many ways, some individuals experience no symptoms, while others may exhibit respiratory symptoms such as fever, cough, and shortness of breath. In certain cases, COVID-19 can cause severe neurological complications, including brain inflammation and nerve damage. It's essential to understand that although vaccines offer strong protection against COVID-19, it's not"}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': 'Yes, it is recommended to

In [126]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.7, .3]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.58 seconds, 17.17 sentences/sec
ROUGE scores: {'rouge1': 0.1464843597751609, 'rouge2': 0.03705757064911994, 'rougeL': 0.10283346135634186, 'rougeLsum': 0.10539253521425472}
BERT scores: {'precision': 0.8281878232955933, 'recall': 0.8734798431396484, 'f1': 0.8501289486885071}
BLEU score: {'bleu': 0.019613081889332244, 'precisions': [0.12525879917184266, 0.02615062761506276, 0.008456659619450317, 0.005341880341880342], 'brevity_penalty': 1.0, 'length_ratio': 3.1161290322580646, 'translation_length': 966, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': "Yes, common symptoms of COVID-19 include fever and cough, along with difficulty breathing, headache, and muscle pain. It's important to note that individuals who have developed severe respiratory symptoms should receive a proper diagnosis before taking any medications or receiving treatment. Additionally, if you experience new symptoms or changes in your condition, seek immediate medical attention. Remember to stay informed and prioritize self-care practices such as washing hands regularly, wearing masks, and practicing social distancing. Stay strong and hopeful!"}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': "\nDoctor: Hello, thank you for reaching out. Let me clarify your question. Do you want to know if wearing a mask is still necessary during everyday activities? My answer would depend on where you live and what local guidelines say. Please check with your local health authorities for t

In [127]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.8, .2]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.59 seconds, 16.84 sentences/sec
ROUGE scores: {'rouge1': 0.17875742076115936, 'rouge2': 0.039731798040680555, 'rougeL': 0.11751509862411566, 'rougeLsum': 0.11939993530808451}
BERT scores: {'precision': 0.8355759382247925, 'recall': 0.8902750015258789, 'f1': 0.8619953393936157}
BLEU score: {'bleu': 0.015046287494665342, 'precisions': [0.12081513828238719, 0.028592375366568914, 0.0066469719350073855, 0.002232142857142857], 'brevity_penalty': 1.0, 'length_ratio': 4.432258064516129, 'translation_length': 1374, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': "It sounds like you would like me to create content for COVID-19, and specifically write a response in English for a question regarding COVID-19 symptoms, given its potential for Neuropsychiatric disorders. Here is my attempt:\n\n1. What are some common symptoms of COVID-19 related neuropsychiatric disorders?\n\nNeuropsychiatric disorders are becoming increasingly common in individuals who contract COVID-19, especially those who experience mild or asymptomatic cases. Some of the most commonly seen symptoms include confusion, psychomotor excitement, poor communication, seizures, mood changes, and cognitive decline. It's important to note that these symptoms can vary widely from person to person and may occur weeks to months after the initial infection.\n\nIf you are experiencing any of these symptoms, it's crucial to speak with a healthcare provider as soon as possible to receive proper evaluation and treatment. They may

In [128]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.9, .1]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.58 seconds, 17.34 sentences/sec
ROUGE scores: {'rouge1': 0.12987539279384896, 'rouge2': 0.03922247264061954, 'rougeL': 0.10034890731071622, 'rougeLsum': 0.10132359688713317}
BERT scores: {'precision': 0.83544921875, 'recall': 0.8770807385444641, 'f1': 0.8554628491401672}
BLEU score: {'bleu': 0.01940120278666314, 'precisions': [0.10233682514101532, 0.02843216896831844, 0.009828009828009828, 0.00495458298926507], 'brevity_penalty': 1.0, 'length_ratio': 4.003225806451613, 'translation_length': 1241, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': '\nHello there! How may I assist you today? Please let me know what specifically you would like to know about COVID-19. Remember, I am here to help and provide you with accurate, up-to-date information in a clear and concise manner.'}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': "\nMask Wearing is Essential!\n\nWearing a mask, especially during physical activities, plays a crucial role in preventing the spread of COVID-19. Your body works harder when you exercise, making it easier for droplets containing the virus to become airborne and travel further. The most effective way to protect yourself and others is by wearing a surgical mask that filters out droplets before they reach the air around you. The results of the research you mentioned indicate that wearing any kind of mask, including a sports-reusable mask, increases breathability perception and causes a slight decrease in exercise perfo

In [129]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[1., 0.]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=50,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.57 seconds, 17.57 sentences/sec
ROUGE scores: {'rouge1': 0.16883663025978035, 'rouge2': 0.04039004479311785, 'rougeL': 0.11517167625282801, 'rougeLsum': 0.1180647623466091}
BERT scores: {'precision': 0.8344415426254272, 'recall': 0.8867262601852417, 'f1': 0.8596989512443542}
BLEU score: {'bleu': 0.02345105825842917, 'precisions': [0.1130030959752322, 0.028081123244929798, 0.013364779874213837, 0.0071315372424722665], 'brevity_penalty': 1.0, 'length_ratio': 4.167741935483871, 'translation_length': 1292, 'reference_length': 310}


[[{'query': '1. "Corona ke symptoms kya hote hain?"',
   'result': "It's common for people to experience various symptoms during COVID-19 infection, including respiratory problems and neurological issues. While it is possible to recover from COVID-19 without any lasting brain damage, some cases have resulted in long-lasting neurological disorders. It's essential to keep in mind that every individual experiences COVID-19 differently, and the severity of symptoms varies widely. Therefore, if you notice significant changes in your mental status, cognitive abilities, or mobility, it's vital to consult with a healthcare professional."}],
 [{'query': '2. "Kya mask pehenna abhi bhi zaroori hai?"',
   'result': "Yes, wearing a mask is generally recommended to help prevent the spread of COVID-19. Research suggests that wearing a mask while exercising may cause some discomfort, but it does not affect physical performance. However, wearing a mask can help reduce the transmission of respiratory dr

In [ ]:
######

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.9, .1]
)

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=1,
    top_k=25,
    top_p=.9,
    repetition_penalty=1.20,
    return_full_text=False,
    max_new_tokens=200,
    truncation=True,
    do_sample=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<s>[INST]
You are an experienced, compassionate COVID-19 doctor. Your patient is speaking in Hindi-English mixed language, but you must respond in clear, simple English. Always provide a direct, helpful answer to the patient's specific question.
Do not ask the patient how you can assist them or request more information unless absolutely necessary.
Understand each word very carefully to give an answer, do not assume anything about the query of the patient.


Key points to remember:
1. Always respond in English, even if the patient uses Hindi or mixed language.
2. Provide a direct answer to the patient's question immediately.
3. Use simple, non-technical language that patients can easily understand.
4. Give practical, actionable advice when appropriate.
5. Be reassuring and empathetic, but honest about health concerns.
6. If you're unsure about a specific treatment or diagnosis, recommend consulting a doctor in person.

Here are two examples of good responses:

Patient: Doctor sahib, mujhe bukhar aur khansi ho rahi hai, kya yeh covid ka lakshan ho sakta hai?
Doctor: Yes, fever and cough can indeed be symptoms of COVID-19. I recommend you get tested for COVID-19 as soon as possible. In the meantime, please isolate yourself at home, rest, and drink plenty of fluids. Monitor your symptoms closely, and if they worsen, especially if you have difficulty breathing, seek medical attention immediately.

Patient: Kya vaccine lagwane ke baad bhi covid ho sakta hai?
Doctor: Yes, it is possible to get COVID-19 even after being vaccinated, although the risk is much lower. If you do get infected after vaccination, your symptoms are likely to be milder. The vaccine significantly reduces your risk of severe illness, hospitalization, and death. However, it's still important to follow safety measures like wearing masks in crowded places and maintaining good hand hygiene.

Now, please respond to the patient's query below in a similar manner, in English:

Patient's question: {question}

Additional context (if available):
{context}

Provide a relevant, caring, and informative response:
[/INST] </s>
"""

Prompt_Template = PromptTemplate(template=prompt_template, input_variables=["question", "context"])

RAG_chain = RetrievalQA.from_chain_type(
    llm=mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

references = []
predictions = []

for p,r in zip(train["Queries "], train["Response"]):
  response = RAG_chain({"query": p})
  references.append([r])
  predictions.append([response])

predictions_fixed = [p[0]['result'] for p in predictions]
references_fixed = [r[0] for r in references]

rouge_scores = rouge.compute(predictions=predictions_fixed, references=references_fixed)
bleu_score = bleu.compute(predictions=predictions_fixed, references=references_fixed)

P, R, F1 = score(predictions_fixed, references_fixed, lang="en", verbose=True)

bert_scores = {"precision": P.mean().item(), "recall": R.mean().item(), "f1": F1.mean().item()}

print("ROUGE scores:", rouge_scores)
print("BERT scores:", bert_scores)
print("BLEU score:", bleu_score)

predictions